## SETUP

In [1]:
from minio import Minio
from dotenv import load_dotenv
from pyspark import SparkContext, SparkConf
import os

load_dotenv()

False

In [2]:
client_minio = Minio(
    f"{os.getenv("minio_ip_address")}:3900",
    access_key=os.getenv("key_id"),
    secret_key=os.getenv("secret_key"),
    secure=False,
    region="garage",
)

conf = SparkConf() \
    .setAppName('SparkApp') \
    .setMaster('spark://spark:7077') \
    .set("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") # utilisé pour le stockage 
sc = SparkContext(conf=conf)

sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint", f"http://{os.getenv('minio_ip_address')}:3900")
sc._jsc.hadoopConfiguration().set("fs.s3a.access.key", os.getenv('key_id')) # set key ID 
sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint.region", "garage")
sc._jsc.hadoopConfiguration().set("fs.s3a.secret.key", os.getenv('secret_key')) # set secret key
sc._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
sc._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
sc._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1c0c9460-98af-415e-85b6-255565d4fdd3;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 117ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------

## Upload train stations to Garage

https://prim.iledefrance-mobilites.fr/fr/jeux-de-donnees/emplacement-des-gares-idf

In [6]:
client_minio.fput_object(os.getenv("bucket_name"), "gares_idf.csv", "/data/emplacement-des-gares-idf.csv")

All columns (in order):  
<table border="0" style="border-collapse: collapse;">
<tr style="border: none;">
<td valign="top" style="border: none; padding-right: 40px;">

0. *geo_point_2d*  
1. *geo_shape*  
2. *objectid_1*  
3. *id_gares*  
4. *nom_gares*  
5. *nom_so_gar*  
6. *nom_su_gar*  
7. *id_ref_zdc*  
8. *nom_zdc*  
9. *id_ref_zda*  

</td>
<td valign="top" style="border: none; padding-right: 40px;">

10. *nom_zda*  
11. *idrefliga*  
12. *idrefligc*  
13. *res_com*  
14. *indice_lig*  
15. *mode*  
16. *tertrain*  
17. *terrer*  
18. *termetro*  
19. *tertram*  

</td>
<td valign="top" style="border: none;">

20. *terval*  
21. *exploitant*  
22. *idf*  
23. *principal*  
24. *x*  
25. *y*  
26. *picto*  
27. *nom_iv*  

</td>
</tr>
</table>

## Gather train stations by line

In [ ]:
bucket = os.getenv("bucket_name")
rdd = sc.textFile(f"s3a://{bucket}/gares_idf.csv")

header = rdd.first()

grouped = (
    rdd
    .filter(lambda line: line != header)
    .map(lambda line: line.split(";"))
    .map(lambda cols: (cols[14], cols[7])) # (line_id, zone_id)
    .partitionBy(8)                        # explicit partitioning
    .groupByKey()
    .mapValues(list)                       # materialize iterable
    .persist()
)

counts = (
    rdd
    .filter(lambda line: line != header)
    .map(lambda line: line.split(";"))
    .map(lambda cols: (cols[14], 1))
    .reduceByKey(lambda a, b: a + b)
    .persist()
)
print(grouped.take(5))
print(counts.take(5))

# DATAFRAME Example
# df_exo = (
#     sql_context.read
#     .format("csv")                  # Specify format as CSV
#     .option("inferSchema", "true")  # Infer the schema automatically
#     .option("header", "true")       # The file contains a header
#     .option("mode", "FAILFAST")     # Fail if there are any errors
#     .option("nullValue", "")        # Treat empty strings as null values
#     .load("s3a://tp6/departuredelays.csv")                 # Path to the CSV file
# )

[('L', ['64741', '73604', '72073', '64251', '72148', '73603', '70686', '73749', '70705', '64382', '71517', '63989', '70713', '64405', '72177', '64341', '71422', '422067', '64301', '72124', '69503', '72132', '64918', '70829', '422420', '66834', '66436', '70956', '73614', '73605', '66696', '66858', '65048', '71370', '71475', '64021']), ('U', ['73749', '63426', '69503', '63812', '73731', '71517', '70829', '71422', '463850', '70686', '63880']), ('4', ['73794', '479068', '71139', '73312', '73360', '73362', '477288', '70441', '71056', '71511', '73653', '71216', '71117', '71202', '73329', '73411', '72033', '72028', '73334', '71410', '71223', '73618', '71184', '70586', '478883', '71359', '411284', '73350', '478648', '73620', '71030', '71556', '73633', '411281', '73354', '477278', '477285', '72646', '71333', '483315', '70657', '71088', '73636', '73709', '73364', '72037', '71426', '71469', '71264']), ('10', ['71166', '71156', '71157', '71170', '71169', '71199', '70163', '71153', '71190', '71150'

## Get trafic news

https://prim.iledefrance-mobilites.fr/fr/apis/idfm-navitia-line_reports-v2

In [24]:
import requests

base_url = "https://prim.iledefrance-mobilites.fr/marketplace/v2/navitia/line_reports"
info_trafic_url = base_url + "/line_reports"
info_trafic_by_line_url = base_url + "/{line_id}/line_reports"

response = requests.get(info_trafic_url,
                        headers={"apiKey": f"{os.getenv("PRIM_API_KEY")}"})
response.raise_for_status()  # Check if the request was successful

data = response.json()
print(data)

{'pagination': {'total_result': 1102, 'start_page': 0, 'items_per_page': 25, 'items_on_page': 25}, 'feed_publishers': [{'id': 'IDFM:idfm_netex', 'name': 'idfm_netex', 'url': '', 'license': 'private'}, {'id': 'IDFM:transilien_netex', 'name': 'transilien_netex', 'url': '', 'license': 'private'}], 'disruptions': [{'id': '71c16bdc-0285-11f1-b66e-0a58a9feac02', 'disruption_id': '71c0fda0-0285-11f1-b66e-0a58a9feac02', 'impact_id': '71c16bdc-0285-11f1-b66e-0a58a9feac02', 'application_periods': [{'begin': '20260225T090000', 'end': '20260306T235900'}], 'publication_period': {'begin': '20260225T090000', 'end': '20260306T235900'}, 'status': 'active', 'updated_at': '20260205T122558', 'tags': ['Actualité'], 'cause': 'travaux', 'category': 'Incidents', 'severity': {'name': 'perturbée', 'effect': 'SIGNIFICANT_DELAYS', 'color': '#EF662F', 'priority': 30}, 'messages': [{'text': '<p><strong>⚠🚍#InfoTrafic - Ligne 25</strong></p><p>&nbsp;</p><p><strong>Vendredi 27 février à partir de 22h00 au samedi 28 fé